# HF-04.5 Batch Input Packager
Packages the preserved observed/aligned pages and reviewed **undilated** HITL seed masks into the ZIP structure expected by HF-04.5.

It searches two run roots recursively:
- the 151-page HITL/V8.9 lineage for `observed.png` + `authoritative_seed_mask.png`;
- the paired-development lineage for `observed.png` + `reviewed_seed.png`.

YOLO is not run and masks are not dilated. Filename prefixes prevent collisions between the two populations.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, json, shutil, zipfile
import cv2

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')

# If auto-discovery selects the wrong run, replace either value with the exact run path.
HITL_RUN_OVERRIDE = None
PAIRED_RUN_OVERRIDE = None

OUT_ROOT = ROOT / 'hf04_5_batch_inputs'
SUPPORTED = {'.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'}

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1024*1024), b''):
            h.update(b)
    return h.hexdigest()

def find_run(root, mask_name, minimum_pages=1):
    hits = []
    if root.exists():
        for p in root.rglob('pages'):
            if not p.is_dir():
                continue
            n = sum(1 for d in p.iterdir() if d.is_dir() and (d/'observed.png').exists() and (d/mask_name).exists())
            if n >= minimum_pages:
                hits.append((n, p.parent.stat().st_mtime, p.parent))
    if not hits:
        return None
    return max(hits, key=lambda x: (x[0], x[1]))[2]

HITL_SEARCH_ROOTS = [ROOT/'v8_9_release', ROOT]
PAIRED_SEARCH_ROOTS = [ROOT/'paired_hitl_development', ROOT]

def resolve(override, roots, mask_name):
    if override:
        p = Path(override)
        if not p.exists(): raise RuntimeError(f'Run override does not exist: {p}')
        return p
    candidates = []
    for r in roots:
        q = find_run(r, mask_name)
        if q is not None:
            pages = q/'pages'
            n = sum(1 for d in pages.iterdir() if d.is_dir() and (d/'observed.png').exists() and (d/mask_name).exists())
            candidates.append((n, q.stat().st_mtime, q))
    return max(candidates, key=lambda x:(x[0],x[1]))[2] if candidates else None

HITL_RUN = resolve(HITL_RUN_OVERRIDE, HITL_SEARCH_ROOTS, 'authoritative_seed_mask.png')
PAIRED_RUN = resolve(PAIRED_RUN_OVERRIDE, PAIRED_SEARCH_ROOTS, 'reviewed_seed.png')

print('151-page HITL run :', HITL_RUN)
print('Paired run        :', PAIRED_RUN)


In [ ]:
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_hf04_5_input_pack'
OUT = OUT_ROOT / RUN_ID
ANN = OUT / 'annotated'
MASK = OUT / 'reviewed_masks'
for d in [ANN, MASK]: d.mkdir(parents=True, exist_ok=False)

rows = []

def collect(run, population, prefix, mask_name):
    if run is None:
        print(f'WARNING: no {population} run found; population omitted')
        return 0
    pages_dir = run/'pages'
    if not pages_dir.is_dir(): raise RuntimeError(f'Missing pages directory: {pages_dir}')
    n = 0
    for d in sorted(p for p in pages_dir.iterdir() if p.is_dir()):
        obs = d/'observed.png'
        msk = d/mask_name
        if not obs.exists() and not msk.exists():
            continue
        if not obs.exists() or not msk.exists():
            raise RuntimeError(f'{population}/{d.name}: incomplete pair')
        oi = cv2.imread(str(obs), cv2.IMREAD_GRAYSCALE)
        mi = cv2.imread(str(msk), cv2.IMREAD_GRAYSCALE)
        if oi is None or mi is None: raise RuntimeError(f'{population}/{d.name}: unreadable image')
        if oi.shape != mi.shape: raise RuntimeError(f'{population}/{d.name}: shape mismatch {oi.shape} vs {mi.shape}')
        if int((mi>0).sum()) == 0: raise RuntimeError(f'{population}/{d.name}: empty reviewed mask')
        stem = f'{prefix}{d.name}'
        out_obs = ANN/f'{stem}.png'
        out_msk = MASK/f'{stem}.png'
        if out_obs.exists() or out_msk.exists(): raise RuntimeError(f'Duplicate output stem: {stem}')
        shutil.copy2(obs, out_obs)
        # Re-save mask as strictly binary uint8 without dilation.
        cv2.imwrite(str(out_msk), ((mi>0)*255).astype('uint8'))
        rows.append({
            'output_stem': stem, 'source_population': population, 'source_page': d.name,
            'source_run': str(run), 'source_observed': str(obs), 'source_mask': str(msk),
            'height': oi.shape[0], 'width': oi.shape[1],
            'reviewed_mask_pixels': int((mi>0).sum()),
            'annotated_sha256': sha256_file(out_obs), 'reviewed_mask_sha256': sha256_file(out_msk),
        })
        n += 1
    return n

n_hitl = collect(HITL_RUN, 'HITL_151', 'HITL_', 'authoritative_seed_mask.png')
n_paired = collect(PAIRED_RUN, 'PAIRED_DEVELOPMENT', 'PAIRED_', 'reviewed_seed.png')

if not rows: raise RuntimeError('No page/mask pairs were collected.')
if len(list(ANN.glob('*.png'))) != len(list(MASK.glob('*.png'))) or len(rows) != len(list(ANN.glob('*.png'))):
    raise RuntimeError('Final annotated/mask roster mismatch.')

with (OUT/'batch_source_manifest.csv').open('w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0]))
    w.writeheader(); w.writerows(rows)

manifest = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'purpose': 'HF-04.5 reviewed-mask A1/A5 batch evaluation input',
    'yolo_bypassed': True,
    'masks_are_reviewed_undilated_seed_masks': True,
    'hitl_run': str(HITL_RUN) if HITL_RUN else None,
    'paired_run': str(PAIRED_RUN) if PAIRED_RUN else None,
    'hitl_pages': n_hitl, 'paired_pages': n_paired, 'total_pages': len(rows),
}
(OUT/'batch_manifest.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))


In [ ]:
# Build exactly the HF-04.5 upload structure.
ZIP_PATH = OUT_ROOT / f'{RUN_ID}.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for p in sorted(ANN.glob('*.png')):
        z.write(p, f'annotated/{p.name}')
    for p in sorted(MASK.glob('*.png')):
        z.write(p, f'reviewed_masks/{p.name}')
    z.write(OUT/'batch_source_manifest.csv', 'batch_source_manifest.csv')
    z.write(OUT/'batch_manifest.json', 'batch_manifest.json')

# Read the ZIP back and fail closed if its paired roster is not exact.
with zipfile.ZipFile(ZIP_PATH) as z:
    names = set(z.namelist())
    a = {Path(n).stem for n in names if n.startswith('annotated/') and n.endswith('.png')}
    m = {Path(n).stem for n in names if n.startswith('reviewed_masks/') and n.endswith('.png')}
    if a != m: raise RuntimeError(f'ZIP paired roster mismatch: annotated-only={sorted(a-m)}, mask-only={sorted(m-a)}')
    if len(a) != len(rows): raise RuntimeError(f'ZIP contains {len(a)} pairs, expected {len(rows)}')

print('\nPACKAGING PASSED')
print('ZIP:', ZIP_PATH)
print('Pairs:', len(rows), f'(HITL={n_hitl}, paired={n_paired})')
print('SHA-256:', sha256_file(ZIP_PATH))
